In [ ]:
!wget http://elm.eu.org/elms/elms_index.tsv -O datasets/elms_classes.tsv
!wget "http://elm.eu.org/instances.tsv?q=*&taxon=&instance_logic=" -O downloads/elms_instances.tsv
!wget "http://elm.eu.org/instances.fasta?q=*&taxon=&instance_logic=" -O downloads/elms_instances.fasta
!wget http://elm.eu.org/goterms.tsv -O downloads/elms_goterms.tsv

'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [79]:
%load_ext autoreload
%autoreload 2

In [23]:
import pandas as pd
elms_classes = pd.read_csv('downloads/elms_classes.tsv', sep='\t', comment='#')
elms_classes.rename(columns={'Accession "ELMIdentifier"': "ELMIdentifier"}, inplace=True)
elms_instances = pd.read_csv('downloads/elms_instances.tsv', sep='\t', comment='#')
elms_terms = pd.read_csv('downloads/elms_goterms.tsv', sep='\t', comment='#')
elms_terms = elms_terms.groupby('ELM')['GOTerm ID'].apply(list).reset_index()

In [24]:
elms_instances = elms_instances.merge(elms_terms, left_on='ELMIdentifier', right_on='ELM', how='left')
elms_instances['AnnotatedIndices'] = [(s, e) for s, e in zip(elms_instances['Start'], elms_instances['End'])]
elms_instances['Regex'] = elms_instances['ELMIdentifier'].map(elms_classes.set_index('ELMIdentifier')['Regex'])

In [27]:
elms_df = elms_instances[['Primary_Acc', 'AnnotatedIndices', 'GOTerm ID', 'ELMIdentifier']].copy()
elms_df.columns = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'ELMIdentifier']
def clean_acc(acc):
    ret = acc
    if ':' in acc:
        ret = acc.split(':')[0]
    if '-' in ret:
        ret = ret.split('-')[0]
    return ret
elms_df['UniprotID'] = elms_df['UniprotID'].apply(clean_acc)
#Remove rows containing nan
elms_df = elms_df.dropna(subset=['UniprotID', 'AnnotatedIndices', 'GOTerm'])

In [29]:
#Aggregate the annotated indices by combining them into a list and grouping by UniprotID
def union_terms(terms):
    r = []
    for term in terms:
        r.extend(term)
    out = list(set(r))
    return out

elms_df['GOTermSet'] = elms_df['GOTerm'].apply(lambda x: ','.join(sorted(x)))
elms_df = elms_df.groupby(['UniprotID', 'GOTermSet']).agg({'AnnotatedIndices': list, 'GOTerm': union_terms, 'ELMIdentifier':'first'}).reset_index().drop(columns='GOTermSet')

In [34]:
take_entry = []
seen = set()
for i, row in elms_df.iterrows():
    elm_entry = row['ELMIdentifier']
    if elm_entry not in seen:
        take_entry.append(True)
        seen.add(elm_entry)
    else:
        take_entry.append(False)
elms_df = elms_df[take_entry].reset_index(drop=True)

In [36]:

from utils import fetch_sequences_from_uniprot_batch
seq_df = elms_df[['UniprotID']].copy()
#Remove duplicate rows from seq df
seq_df = seq_df.drop_duplicates()

In [37]:
seq_df['Sequence'] = [x for _, x in fetch_sequences_from_uniprot_batch(seq_df['UniprotID'].tolist())]

304
Sending batch query for 304 accessions...
Processed 304 accessions, out of 304 total.


In [38]:
elms_df = elms_df.merge(seq_df, on='UniprotID', how='left')
elms_df.dropna()
elms_df.to_csv('datasets/elms_dataset.csv', index=False, sep='\t')

In [ ]:
# import pandas as pd
# dataset_df = pd.read_csv('datasets/elms_dataset.csv', sep='\t').dropna()
elms_df = elms_df[[len(t) < 850 for t in elms_df['Sequence']]]
elms_df.to_csv('datasets/elms_dataset.csv', index=False, sep='\t')